In [1]:
import numpy as np
import WDM

from WDM.code.time_delay_filters.affine_filter_table import AffineFilterTable
from WDM.code.time_delay_filters.affine_sparse_operator import AffineSparseOperator



In [2]:
wdm = WDM.WDM.WDM_transform(
    dt=10.0,
    Nf=1024,
    N=1024 * 1024,
    q=64,
    d=4,
    A_frac=0.25,
    calc_m0=False,
)

In [4]:
ell_values = np.array([-3, -1, 0, 1, 3], dtype=int)

D_grid = np.linspace(-600.0, 600.0, 33)
eps_grid = np.linspace(-1.2e-4, 1.2e-4, 17)
eta_grid = np.linspace(-0.15, 0.15, 129)

table = WDM.affine_filter_table.AffineFilterTable.build_or_load(
    "affine_filter_table_branch_symmetric_test2.pkl",
    wdm,
    D_grid=D_grid,
    eps_grid=eps_grid,
    eta_grid=eta_grid,
    ell_values=ell_values,
    sigma_values=np.array([-1, 0, +1], dtype=int),
    use_branch_symmetry=True,
    n_quad_table=2048,
    lambda_chunk_size=256,
    rebuild=False,
    check_build_parameters=True,
    verbose=True,
)

Building affine filter table
500/6579
1000/6579
1500/6579
2000/6579
2500/6579
3000/6579
3500/6579
4000/6579
4500/6579
5000/6579
5500/6579
6000/6579
6500/6579
6579/6579
Saved affine filter table to affine_filter_table_branch_symmetric_test2.pkl


In [5]:
operator = WDM.affine_sparse_operator.AffineSparseOperator(wdm, table)

print("loaded successfully")
print("num stored tables:", len(table.Q_tables))
print("stored rho values:", table.rho_values_stored)
print("use_branch_symmetry:", table.use_branch_symmetry)

loaded successfully
num stored tables: 15
stored rho values: [1]
use_branch_symmetry: True


In [6]:

rng = np.random.default_rng(1234)

w_in = np.zeros((wdm.Nt, wdm.Nf), dtype=np.complex128)
w_in[:, 1:] = rng.normal(size=(wdm.Nt, wdm.Nf - 1))

n_array = np.arange(wdm.Nt)

D_of_n = 200.0 * np.sin(2.0 * np.pi * n_array / wdm.Nt)
epsilon_of_n = 5.0e-5 * np.sin(2.0 * np.pi * n_array / wdm.Nt + 0.3)

n_targets = np.arange(wdm.Nt // 2 - 4, wdm.Nt // 2 + 5)
m_targets = np.arange(50, 61)

w_out_selected = operator.apply_selected(
    w_in,
    D_of_n=D_of_n,
    epsilon_of_n=epsilon_of_n,
    n_targets=n_targets,
    m_targets=m_targets,
)

print(w_out_selected.shape)
print(np.abs(w_out_selected).min(), np.abs(w_out_selected).max())

(9, 11)
0.003418206494285538 2.776139532402594


In [7]:
from WDM_LISAresponse.code.validation.affine_filter_reference import (
    branch_summed_affine_entry,
)

import pandas as pd

In [10]:
rng = np.random.default_rng(777)

n_tests = 500
rows = []

n0 = int(wdm.Nt // 2)

for i in range(n_tests):
    ell = int(rng.choice(table.ell_values))
    sigma = int(rng.choice(table.sigma_values))

    for attempt in range(1000):
        m = int(rng.integers(2, wdm.Nf - 2))
        epsilon = rng.uniform(
            table.eps_grid[0],
            table.eps_grid[-1],
        )
        eta = epsilon * (m + sigma)

        if (
            1 <= m + sigma < wdm.Nf
            and table.eta_grid[0] <= eta <= table.eta_grid[-1]
        ):
            break
    else:
        raise RuntimeError("Could not find valid m/epsilon/eta.")

    D = rng.uniform(
        table.D_grid[0],
        table.D_grid[-1],
    )

    module_val = table.evaluate_K(
        wdm,
        n=n0,
        ell=ell,
        m=m,
        sigma=sigma,
        D=D,
        epsilon=epsilon,
    )

    direct_val = branch_summed_affine_entry(
        wdm,
        n=n0,
        ell=ell,
        m=m,
        sigma=sigma,
        D=D,
        epsilon=epsilon,
        n_quad=4096,
        return_details=False,
    )

    abs_diff = abs(module_val.real - direct_val.real)
    amp = max(abs(module_val.real), abs(direct_val.real))
    rel_diff = abs_diff / max(amp, 1e-300)

    rows.append({
        "i": i,
        "ell": ell,
        "sigma": sigma,
        "m": m,
        "D": D,
        "epsilon": epsilon,
        "eta": eta,
        "module_real": module_val.real,
        "direct_real": direct_val.real,
        "module_imag": module_val.imag,
        "direct_imag": direct_val.imag,
        "amp": amp,
        "abs_diff": abs_diff,
        "rel_diff": rel_diff,
    })

df_module_K_check = pd.DataFrame(rows)

print("median abs diff:", df_module_K_check["abs_diff"].median())
print("max abs diff:   ", df_module_K_check["abs_diff"].max())
print("median rel diff:", df_module_K_check["rel_diff"].median())
print("max rel diff:   ", df_module_K_check["rel_diff"].max())

df_module_K_check.sort_values("rel_diff", ascending=False).head(20)

median abs diff: 1.2316368240040954e-06
max abs diff:    1.1861822820868895e-05
median rel diff: 2.1942519680535873e-05
max rel diff:    0.00019648042709331334


,i,ell,sigma,m,D,epsilon,eta,module_real,direct_real,module_imag,direct_imag,amp,abs_diff,rel_diff
370,370,1,-1,970,124.416581,-0.000118,-0.114097,-0.011205,-0.011203,1.734928e-18,-1.734928e-18,0.011205,2.201514e-06,0.000196
322,322,0,-1,989,321.599753,-0.000105,-0.103968,0.003956,0.003956,-1.734906e-18,0.000000e+00,0.003956,7.231029e-07,0.000183
276,276,-3,-1,994,-387.121421,-0.000110,-0.109075,0.016590,0.016588,0.000000e+00,8.674570e-19,0.016590,2.691311e-06,0.000162
62,62,-1,1,933,145.669531,0.000094,0.087446,0.028573,0.028569,3.469122e-18,0.000000e+00,0.028573,3.984632e-06,0.000139
161,161,-3,1,892,-167.690443,0.000114,0.101412,0.024225,0.024222,2.168158e-19,0.000000e+00,0.024225,3.092224e-06,0.000128
475,475,-1,-1,733,344.833150,-0.000118,-0.086259,0.031564,0.031560,8.674640e-19,0.000000e+00,0.031564,3.791009e-06,0.000120
75,75,1,1,904,-336.461472,0.000094,0.084798,-0.023838,-0.023835,6.938244e-18,0.000000e+00,0.023838,2.733596e-06,0.000115
129,129,-1,1,743,-90.418325,0.000119,0.088764,0.032985,0.032982,2.168146e-19,0.000000e+00,0.032985,3.580824e-06,0.000109
251,251,-3,-1,740,249.071862,-0.000120,-0.088389,-0.012760,-0.012759,-3.469862e-18,3.469862e-18,0.012760,1.364401e-06,0.000107
424,424,-1,1,705,134.797216,0.000088,0.062240,0.038927,0.038923,-5.203712e-18,0.000000e+00,0.038927,4.150214e-06,0.000107


In [11]:
def direct_affine_sparse_entry_reference(
    wdm,
    w_in,
    *,
    n,
    m,
    D,
    epsilon,
    ell_values,
    sigma_values,
    n_quad=4096,
):
    total = 0.0 + 0.0j

    n = int(n)
    m = int(m)

    for ell in ell_values:
        ell = int(ell)
        n_source = n - ell

        if not (0 <= n_source < wdm.Nt):
            continue

        for sigma in sigma_values:
            sigma = int(sigma)
            m_source = m + sigma

            if not (1 <= m_source < wdm.Nf):
                continue

            K = branch_summed_affine_entry(
                wdm,
                n=n,
                ell=ell,
                m=m,
                sigma=sigma,
                D=D,
                epsilon=epsilon,
                n_quad=n_quad,
                return_details=False,
            )

            total += K * w_in[n_source, m_source]

    return complex(total)

In [12]:
rng = np.random.default_rng(778)

n_checks = 50
rows = []

for i in range(n_checks):
    in_idx = int(rng.integers(0, len(n_targets)))
    im_idx = int(rng.integers(0, len(m_targets)))

    n = int(n_targets[in_idx])
    m = int(m_targets[im_idx])

    D = float(D_of_n[n])
    epsilon = float(epsilon_of_n[n])

    module_sparse = w_out_selected[in_idx, im_idx]

    direct_sparse = direct_affine_sparse_entry_reference(
        wdm,
        w_in,
        n=n,
        m=m,
        D=D,
        epsilon=epsilon,
        ell_values=table.ell_values,
        sigma_values=table.sigma_values,
        n_quad=4096,
    )

    abs_diff = abs(module_sparse - direct_sparse)
    amp = max(abs(module_sparse), abs(direct_sparse))
    rel_diff = abs_diff / max(amp, 1e-300)

    rows.append({
        "i": i,
        "n": n,
        "m": m,
        "D": D,
        "epsilon": epsilon,
        "eta_nominal": epsilon * m,
        "module_sparse": module_sparse,
        "direct_sparse": direct_sparse,
        "amp": amp,
        "abs_diff": abs_diff,
        "rel_diff": rel_diff,
    })

df_module_sparse_check = pd.DataFrame(rows)

print("median abs diff:", df_module_sparse_check["abs_diff"].median())
print("max abs diff:   ", df_module_sparse_check["abs_diff"].max())
print("median rel diff:", df_module_sparse_check["rel_diff"].median())
print("max rel diff:   ", df_module_sparse_check["rel_diff"].max())

df_module_sparse_check.sort_values("rel_diff", ascending=False).head(20)

median abs diff: 5.615160651228912e-06
max abs diff:    2.960400318663403e-05
median rel diff: 1.0420758484343609e-05
max rel diff:    3.456136653976524e-05


,i,n,m,D,epsilon,eta_nominal,module_sparse,direct_sparse,amp,abs_diff,rel_diff
35,35,513,52,-1.227177,-0.000015,-0.000784,0.013490-0.000000j,0.013490-0.000000j,0.013490,4.662386e-07,0.000035
14,14,516,59,-4.908246,-0.000016,-0.000941,0.324278-0.000000j,0.324283+0.000000j,0.324283,4.906180e-06,0.000015
15,15,516,59,-4.908246,-0.000016,-0.000941,0.324278-0.000000j,0.324283+0.000000j,0.324283,4.906180e-06,0.000015
44,44,509,58,3.681346,-0.000014,-0.000806,0.494346-0.000000j,0.494353+0.000000j,0.494353,6.606863e-06,0.000013
40,40,513,53,-1.227177,-0.000015,-0.000799,-0.003418-0.000000j,-0.003418-0.000000j,0.003418,4.429964e-08,0.000013
36,36,513,53,-1.227177,-0.000015,-0.000799,-0.003418-0.000000j,-0.003418-0.000000j,0.003418,4.429964e-08,0.000013
34,34,508,56,4.908246,-0.000014,-0.000762,0.323860-0.000000j,0.323864+0.000000j,0.323864,3.776885e-06,0.000012
31,31,508,56,4.908246,-0.000014,-0.000762,0.323860-0.000000j,0.323864+0.000000j,0.323864,3.776885e-06,0.000012
9,9,515,53,-3.681346,-0.000016,-0.000830,0.765241-0.000000j,0.765250+0.000000j,0.765250,8.922106e-06,0.000012
41,41,515,55,-3.681346,-0.000016,-0.000861,0.494487-0.000000j,0.494493-0.000000j,0.494493,5.706961e-06,0.000012


In [ ]:
summary_module_sparse = (
    df_module_sparse_check
    .assign(significant=df_module_sparse_check["amp"] > 1e-6)
    .groupby("significant")
    .agg(
        num_cases=("rel_diff", "size"),
        median_amp=("amp", "median"),
        min_amp=("amp", "min"),
        max_amp=("amp", "max"),
        median_abs_diff=("abs_diff", "median"),
        max_abs_diff=("abs_diff", "max"),
        median_rel_diff=("rel_diff", "median"),
        max_rel_diff=("rel_diff", "max"),
    )
    .reset_index()
)

summary_module_sparse

,significant,num_cases,median_amp,min_amp,max_amp,median_abs_diff,max_abs_diff,median_rel_diff,max_rel_diff
0,True,50,0.533996,0.003418,2.776169,0.000006,0.00003,0.00001,0.000035
